# 08 · Métrica v4 — Cover Shadow Value (CSV)

Fase 8c do CRISP-DM (D38). A v3 mantém-se intacta e reproduzível; a v4 é uma versão nova que pesa cada linha de passe pela probabilidade de ser escolhida (o M2 da fase 8b, `07_modelo_escolha.ipynb`) e separa duas formas de a sombra tirar perigo:

- **interceção** (I): o passe é feito e a bola não chega;
- **dissuasão** (D): o portador muda de escolha por causa da sombra.

Em cada momento alinhado da posse (5 por segundo), com C o conjunto de defesas que projetam sombra (não marcam, têm pose e podem afetar alguma linha):

| símbolo | definição |
|---|---|
| `p_l(S)` | probabilidade de o passe da linha l chegar com as sombras da coligação S ⊆ C (como na v3); `p_l(∅)` = 1 |
| `s_l(S)` | probabilidade de escolha da linha l no M2, com `sombra_tronco` = 1 − `p_l(S)`; coeficientes ajustados no outro jogo |
| `x_l` | `valor_xt` da linha |
| `E(S)` | `Σ s_l(S) · p_l(S) · x_l` (perigo esperado) |
| `v(S)` | `E(∅) − E(S)` (ameaça negada) |
| `I(S)`, `D(S)` | interceção e dissuasão; a decomposição de `v` depende da ordem em que se mudam os dois fatores (secção 7): ordem 1, ordem 2 ou simétrica |

Linhas: as da v3, sem fora de jogo (D26), sem as da D27 e sem linhas com mais de 35 m (D37); as linhas com algum defesa sem pose entram com a sombra dos defesas com pose e `linha_sem_pose` = 1. O Shapley de I e de D reparte-se pelos defesas de C por enumeração exata das coligações. Agregação como na v3 (média por posse, soma por jogador); a ameaça disponível é `E(∅)`. Unidade: xT esperado.

Decomposição de `v` em I e D (D38), com `p_l(∅) = 1`:

| versão | interceção `I(S)` | dissuasão `D(S)` |
|---|---|---|
| ordem 1 | `Σ s_l(∅) · (1 − p_l(S)) · x_l` | `Σ (s_l(∅) − s_l(S)) · p_l(S) · x_l` |
| ordem 2 | `Σ s_l(S) · (1 − p_l(S)) · x_l` | `Σ (s_l(∅) − s_l(S)) · x_l` |
| simétrica | `(I₁ + I₂) / 2` | `(D₁ + D₂) / 2` |

Em todas, `I + D = v`. Nas secções 1, 2 e 4 a 6, `I` e `D` são os da versão simétrica (a omissão do código); a secção 3 e a secção 7 mostram as três lado a lado.


In [1]:
import sys

import pandas as pd
from IPython.display import Markdown, display

from footy.ingest import skillcorner as sc
from footy.paths import PROJECTS

sys.path.insert(0, str(PROJECTS / "cover_shadow"))  # para importar o código do projeto
import analise_fundo as af
import avaliacao as av
import calcular_v4 as c4
from metrica.parametros import PADRAO
from metrica.pipeline import carregar_tabela, preparar_defesas

pd.options.display.float_format = "{:.4f}".format
JOGOS = list(sc.JOGOS_COM_POSE)
NOMES = {1925299: "Brisbane Roar – Perth Glory", 1996435: "Sydney FC – Adelaide United"}
V4 = {j: c4.ler_v4(j, "sem_fundo") for j in JOGOS}  # a v4 da D38 (secções 1 a 7); a do fundo (D39) é a secção 8
V4F = {j: c4.ler_v4(j) for j in JOGOS}
V4G = {j: c4.ler_v4(j, "sem_guarda_redes") for j in JOGOS}
MATCH = {j: sc.ler_match(j) for j in JOGOS}
JOGADORES = {j: c4.jogadores_do_jogo(MATCH[j]) for j in JOGOS}
DEFESAS = {j: preparar_defesas(carregar_tabela(j), MATCH[j], PADRAO) for j in JOGOS}

## 1. Testes e identidades

Testes unitários em `tests/test_v4.py` (`uv run pytest projects/cover_shadow/tests/test_v4.py`): identidade I + D = v em todas as coligações e nas três ordens; a simétrica é a média das duas ordens (também no Shapley); eficiência do Shapley (soma = v(C), para I e para D); sem sombras, tudo zero; um exemplo feito à mão com 2 defesas e 3 linhas (Shapley de I = 0,0375 e 0,2375) e outro com dissuasão negativa (ordem 1: I = 0,5, D = −0,25; ordem 2: I = 0,25, D = 0; simétrica: I = 0,375, D = −0,125; v = 0,25).

Nos dados reais, erros máximos (em xT) das mesmas identidades, sobre todos os momentos:

In [2]:
erros = pd.DataFrame({j: c4.verificar_identidades(V4[j]) for j in JOGOS})
display(erros.rename(columns=lambda j: str(j)).map("{:.2e}".format))

,1925299,1996435
I_mais_D_menos_v,5.55e-17,4.16e-17
shapley_I_menos_I,2.78e-17,2.78e-17
shapley_D_menos_D,1.39e-17,7.81e-18
soma_das_linhas_menos_v,2.78e-17,4.16e-17


## 2. Resumo por jogo

Somas sobre todos os momentos, em xT esperado. `n_defesas` = defesas em C por momento; os momentos com mais de 10 defesas em C teriam ficado só com os 10 que mais perigo cobrem.

In [3]:
linhas_resumo = {}
for j in JOGOS:
    m, l = V4[j]["momentos"], V4[j]["linhas"]
    linhas_resumo[j] = {"momentos": len(m), "linhas": len(l), "linhas com `linha_sem_pose`": int(l["linha_sem_pose"].sum()),
                        "momentos com algum defesa em C": int((m["n_defesas"] > 0).sum()),
                        "n_defesas (máx.)": int(m["n_defesas"].max()), "momentos com > 10 defesas": int((m["n_defesas_cortados"] > 0).sum()),
                        "E(∅) total": m["E0"].sum(), "E(C) total": m["EC"].sum(), "I total": m["I"].sum(), "D total": m["D"].sum(),
                        "v total": m["v"].sum(), "v / E(∅) (%)": 100 * m["v"].sum() / m["E0"].sum()}
display(pd.DataFrame(linhas_resumo).rename(columns=lambda j: f"{j} ({NOMES[j]})"))

,1925299 (Brisbane Roar – Perth Glory),1996435 (Sydney FC – Adelaide United)
momentos,8915.0000,8145.0000
linhas,63151.0000,57866.0000
linhas com `linha_sem_pose`,15388.0000,9324.0000
momentos com algum defesa em C,8276.0000,7600.0000
n_defesas (máx.),9.0000,10.0000
momentos com > 10 defesas,0.0000,0.0000
E(∅) total,46.5613,40.7382
E(C) total,26.2249,28.1935
I total,16.1198,10.7940
D total,4.2167,1.7506


## 3. Lances C06 e C03

E(∅), E(C), I, D e Shapley de cada defesa (I, D e total) nas três versões da decomposição, ao lado da v3 (ameaça disponível e negada, e Shapley da v3). Na v3 a ameaça é uma soma de alternativas; na v4 é um xT esperado.

In [4]:
ORDENS = ("ordem1", "ordem2", "simetrica")
for rotulo, jogo, frame in (("C06", 1925299, 12660), ("C03", 1996435, 39646)):
    totais, por_defesa = c4.lance_v3_e_v4(jogo, frame, V4[jogo], JOGADORES[jogo], ordens=ORDENS)
    display(Markdown(f"**{rotulo}** · jogo {jogo} · frame {frame}"))
    display(totais)
    display(por_defesa)

**C06** · jogo 1925299 · frame 12660

,v3,v4 ordem1,v4 ordem2,v4 simetrica
linhas avaliadas,10.0000,7.0000,7.0000,7.0000
disponível,0.0344,0.0029,0.0029,0.0029
E(∅),NaN,0.0029,0.0029,0.0029
E(C),NaN,0.0026,0.0026,0.0026
ameaça negada,0.0120,0.0003,0.0003,0.0003
interceção (I),NaN,0.0003,0.0001,0.0002
dissuasão (D),NaN,-0.0000,0.0002,0.0001


,v3,v4 ordem1 I,v4 ordem1 D,v4 ordem1 total,v4 ordem2 I,v4 ordem2 D,v4 ordem2 total,v4 simetrica I,v4 simetrica D,v4 simetrica total
defesa,,,,,,,,,,
K. Jelacic,0.0043,0.0003,-0.0000,0.0003,0.0001,0.0002,0.0003,0.0002,0.0001,0.0003
F. Bérenguer,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
A. Zimarino,0.0074,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


**C03** · jogo 1996435 · frame 39646

,v3,v4 ordem1,v4 ordem2,v4 simetrica
linhas avaliadas,9.0000,8.0000,8.0000,8.0000
disponível,0.0274,0.0018,0.0018,0.0018
E(∅),NaN,0.0018,0.0018,0.0018
E(C),NaN,0.0012,0.0012,0.0012
ameaça negada,0.0072,0.0006,0.0006,0.0006
interceção (I),NaN,0.0007,0.0002,0.0004
dissuasão (D),NaN,-0.0001,0.0004,0.0002


,v3,v4 ordem1 I,v4 ordem1 D,v4 ordem1 total,v4 ordem2 I,v4 ordem2 D,v4 ordem2 total,v4 simetrica I,v4 simetrica D,v4 simetrica total
defesa,,,,,,,,,,
P. Klimala,0.0042,0.0007,-0.0001,0.0006,0.0002,0.0004,0.0006,0.0004,0.0002,0.0006
J. Lolley,0.0030,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


## 4. Por jogador: v3 e v4

Ameaça negada por jogador (soma das médias por posse, em xT). Com a decomposição simétrica. ρ de Spearman entre a v3 e a v4 dentro de cada posição (posições com pelo menos 4 jogadores nos dois jogos juntos; média ponderada pelo número de jogadores). `parte_I` e `parte_D` = fração da ameaça negada da v4 que vem da interceção e da dissuasão.

In [5]:
jogadores = pd.concat([c4.jogadores_v3_e_v4(j, V4[j], DEFESAS[j], JOGADORES[j]) for j in JOGOS], ignore_index=True)
base = jogadores[["jogo", "jogador", "posicao", "negada_v3"]].rename(columns={"negada_v3": "negada"})
for nome, coluna in (("v4 total", "negada_v4"), ("v4 só interceção (I)", "negada_I")):
    r = av.spearman_por_posicao(base, jogadores[["jogo", "jogador", "posicao", coluna]].rename(columns={coluna: "negada"}))
    display(Markdown(f"**ρ de Spearman, v3 vs {nome}** · média ponderada {r['rho_media']:.3f} · mínimo {r['rho_min']:.3f}"))
    display(r["posicoes"].set_index("posicao"))

**ρ de Spearman, v3 vs v4 total** · média ponderada 0.827 · mínimo 0.400

,n,rho
posicao,,
AM,5,1.0000
CF,9,0.9333
LB,4,1.0000
LCB,5,1.0000
LW,8,0.5476
RB,5,0.9000
RCB,4,0.4000
RDM,5,0.7000
RW,8,0.9286


**ρ de Spearman, v3 vs v4 só interceção (I)** · média ponderada 0.875 · mínimo 0.400

,n,rho
posicao,,
AM,5,1.0000
CF,9,0.9333
LB,4,1.0000
LCB,5,0.9000
LW,8,0.9286
RB,5,0.9000
RCB,4,0.4000
RDM,5,0.7000
RW,8,0.9286


In [6]:
jogadores["parte_I"] = jogadores["negada_I"] / jogadores["negada_v4"]
jogadores["parte_D"] = jogadores["negada_D"] / jogadores["negada_v4"]
nomes = pd.concat([JOGADORES[j].assign(jogo=j) for j in JOGOS])[["jogo", "jogador", "nome"]]
colunas = ["jogo", "nome", "posicao", "negada_v3", "negada_v4", "negada_I", "negada_D", "parte_I", "parte_D", "pct_negada"]
tabela = jogadores.merge(nomes, on=["jogo", "jogador"])[colunas].sort_values("negada_v4", ascending=False, ignore_index=True)
display(Markdown("**Jogadores com maior ameaça negada na v4** (primeiros 20; `pct_negada` = % de E(∅) negada)"))
display(tabela.head(20))
display(Markdown("**Soma por jogo** (xT)"))
display(jogadores.groupby("jogo")[["negada_v3", "negada_v4", "negada_I", "negada_D"]].sum())

**Jogadores com maior ameaça negada na v4** (primeiros 20; `pct_negada` = % de E(∅) negada)

,jogo,nome,posicao,negada_v3,negada_v4,negada_I,negada_D,parte_I,parte_D,pct_negada
0,1925299,N. Pennington,LM,2.1889,0.3693,0.2736,0.0958,0.7407,0.2593,10.7625
1,1925299,A. Lebib,RCB,0.7804,0.2075,0.1468,0.0607,0.7074,0.2926,6.0467
2,1925299,T. Gomulka,RM,1.5137,0.1644,0.1232,0.0412,0.7494,0.2506,4.7911
3,1925299,T. Mrčela,LCB,0.9940,0.1619,0.1034,0.0585,0.6389,0.3611,4.7174
4,1996435,R. Grant,RCB,0.3007,0.1467,0.1020,0.0448,0.6950,0.3050,7.4577
5,1996435,P. Kikianis,LCB,0.6501,0.1439,0.1053,0.0385,0.7321,0.2679,6.1361
6,1996435,E. Alagich,LDM,1.3158,0.1209,0.0932,0.0277,0.7710,0.2290,5.1569
7,1925299,J. Risdon,RB,0.7277,0.1140,0.1049,0.0091,0.9201,0.0799,3.3228
8,1996435,F. Talladira,LB,0.6405,0.1004,0.0867,0.0136,0.8642,0.1358,4.2813
9,1996435,K. Shaw,LCB,0.2691,0.0987,0.0712,0.0275,0.7217,0.2783,5.0164


**Soma por jogo** (xT)

,negada_v3,negada_v4,negada_I,negada_D
jogo,,,,
1925299,12.2001,1.7538,1.4401,0.3137
1996435,9.9080,1.2052,1.0412,0.1640


## 5. Distribuição da dissuasão (D)

D pode ser negativa: a sombra de um defesa pode empurrar o portador para linhas mais perigosas que ficaram livres. Contagem de momentos (com algum defesa em C) por sinal de D, e resumo da distribuição.

In [7]:
m = pd.concat([V4[j]["momentos"].assign(jogo=j) for j in JOGOS])
m = m[m["n_defesas"] > 0]
eps = 1e-9


def resumo_de_d(g):
    return {"momentos com defesas em C": len(g), "D < 0": int((g["D"] < -eps).sum()), "D ≈ 0": int((g["D"].abs() <= eps).sum()),
            "D > 0": int((g["D"] > eps).sum()), "% com D < 0": 100 * (g["D"] < -eps).mean(),
            "soma de D < 0": g.loc[g["D"] < -eps, "D"].sum(), "soma de D > 0": g.loc[g["D"] > eps, "D"].sum()}


display(pd.DataFrame({**{j: resumo_de_d(g) for j, g in m.groupby("jogo")}, "total": resumo_de_d(m)}))
display(Markdown("**Quantis de D** (xT esperado por momento)"))
display(m["D"].quantile([0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1]).to_frame("D"))
display(Markdown("**D / v** nos momentos com v > 0 (fração da ameaça negada que vem da dissuasão)"))
com_v = m[m["v"] > eps]
display((com_v["D"] / com_v["v"]).describe().to_frame("D / v"))

,1925299,1996435,total
momentos com defesas em C,8276.0000,7600.0000,15876.0000
D < 0,2443.0000,2769.0000,5212.0000
D ≈ 0,235.0000,342.0000,577.0000
D > 0,5598.0000,4489.0000,10087.0000
% com D < 0,29.5191,36.4342,32.8294
soma de D < 0,-1.0461,-1.4310,-2.4772
soma de D > 0,5.2628,3.1816,8.4444


**Quantis de D** (xT esperado por momento)

,D
0.0000,-0.0360
0.0100,-0.0031
0.0500,-0.0005
0.2500,-0.0000
0.5000,0.0000
0.7500,0.0002
0.9500,0.0018
0.9900,0.0138
1.0000,0.0417


**D / v** nos momentos com v > 0 (fração da ameaça negada que vem da dissuasão)

,D / v
count,12090.0000
mean,-0.0720
std,3.9544
min,-202.4993
25%,0.1501
50%,0.3443
75%,0.4421
max,0.6322


## 6. Linhas sem pose

Parte da ameaça negada (v) que vem de linhas sem pose (`linha_sem_pose` = 1, sombra só com os defesas com pose) e de momentos com pelo menos uma dessas linhas. A v de cada linha é `x_l · (s_l(∅) − s_l(C) · p_l(C))`; a soma por momento dá v.

In [8]:
saida = {}
for j in JOGOS:
    l, m = V4[j]["linhas"], V4[j]["momentos"]
    sem_pose = l.loc[l["linha_sem_pose"] == 1, "negada_linha"].sum()
    momentos_sem_pose = m[m["n_linhas_sem_pose"] > 0]
    saida[j] = {"linhas": len(l), "% de linhas sem pose": 100 * l["linha_sem_pose"].mean(),
                "momentos": len(m), "% de momentos com alguma linha sem pose": 100 * len(momentos_sem_pose) / len(m),
                "v total": m["v"].sum(), "v das linhas sem pose": sem_pose, "% de v das linhas sem pose": 100 * sem_pose / m["v"].sum(),
                "v dos momentos com alguma linha sem pose": momentos_sem_pose["v"].sum(),
                "% de v dos momentos com alguma linha sem pose": 100 * momentos_sem_pose["v"].sum() / m["v"].sum()}
display(pd.DataFrame(saida).rename(columns=lambda j: f"{j} ({NOMES[j]})"))

,1925299 (Brisbane Roar – Perth Glory),1996435 (Sydney FC – Adelaide United)
linhas,63151.0000,57866.0000
% de linhas sem pose,24.3670,16.1131
momentos,8915.0000,8145.0000
% de momentos com alguma linha sem pose,64.0045,45.0215
v total,20.3364,12.5446
v das linhas sem pose,1.9077,0.9343
% de v das linhas sem pose,9.3806,7.4478
v dos momentos com alguma linha sem pose,9.8220,6.0147
% de v dos momentos com alguma linha sem pose,48.2974,47.9462


## 7. As três decomposições lado a lado

Totais de `I`, `D` e `v` por jogo (xT esperado), e contagem dos momentos com algum defesa em C em que `D < 0`. `v` é igual nas três; só muda a divisão entre `I` e `D`.

In [9]:
eps = 1e-9
tot, sinal = {}, {}
for j in JOGOS:
    m = V4[j]["momentos"]
    com_c = m[m["n_defesas"] > 0]
    for o in ORDENS:
        tot[(j, o)] = {"I total": m[f"I_{o}"].sum(), "D total": m[f"D_{o}"].sum(), "v total": m["v"].sum(),
                       "I + D − v (máx. abs.)": (m[f"I_{o}"] + m[f"D_{o}"] - m["v"]).abs().max()}
        d = com_c[f"D_{o}"]
        sinal[(j, o)] = {"momentos com defesas em C": len(d), "D < 0": int((d < -eps).sum()), "D ≈ 0": int((d.abs() <= eps).sum()),
                         "D > 0": int((d > eps).sum()), "% com D < 0": 100 * (d < -eps).mean()}
display(Markdown("**Totais por jogo**"))
display(pd.DataFrame(tot).T.rename_axis(["jogo", "versão"]))
display(Markdown("**Sinal de D por momento**"))
display(pd.DataFrame(sinal).T.rename_axis(["jogo", "versão"]))

**Totais por jogo**

I total  D total  v total  I + D − v (máx. abs.)
jogo    versão                                                     
1925299 ordem1     21.0722  -0.7357  20.3364                 0.0000
        ordem2     11.1674   9.1691  20.3364                 0.0000
        simetrica  16.1198   4.2167  20.3364                 0.0000
1996435 ordem1     14.1186  -1.5739  12.5446                 0.0000
        ordem2      7.4695   5.0751  12.5446                 0.0000
        simetrica  10.7940   1.7506  12.5446                 0.0000

**Sinal de D por momento**

momentos com defesas em C     D < 0    D ≈ 0     D > 0  \
jogo    versão                                                              
1925299 ordem1                     8276.0000 4061.0000 330.0000 3885.0000   
        ordem2                     8276.0000 2012.0000 235.0000 6029.0000   
        simetrica                  8276.0000 2443.0000 235.0000 5598.0000   
1996435 ordem1                     7600.0000 4196.0000 435.0000 2969.0000   
        ordem2                     7600.0000 2272.0000 354.0000 4974.0000   
        simetrica                  7600.0000 2769.0000 342.0000 4489.0000   

                   % com D < 0  
jogo    versão                  
1925299 ordem1         49.0696  
        ordem2         24.3113  
        simetrica      29.5191  
1996435 ordem1         55.2105  
        ordem2         29.8947  
        simetrica      36.4342

## 8. Fundo (D39): defesas que cortam o passe sem projetar sombra

Nas secções 1 a 7 (a v4 da D38), uma linha sem sombra chega com `p = 1`, mesmo com o recetor marcado. A secção 8 acrescenta o **fundo**
F = os defesas que não estão em C (a marcar, sem pose, e o guarda-redes): não recebem crédito (D5, D25) mas podem cortar o passe.

- `p_fundo_l`: probabilidade de o passe da linha l chegar considerando só F, com o mesmo modelo de tempo de interceção (Spearman/Shaw,
  mesmos parâmetros), **sem cone nem bloqueio pelo corpo**; independentes, como na v3.
- `p_l(C) = p_fundo_l × p_sombras_l(C)` e `p_l(∅) = p_fundo_l`. A escolha (`s_l`) continua a depender só das sombras, com os mesmos coeficientes do M2.
- «sem fundo» = a v4 da D38 (`v4_sem_fundo/`); «com fundo» = `v4/`; «sem o guarda-redes» = F sem o guarda-redes (`v4_sem_guarda_redes/`).

Só números; nenhuma leitura.

### 8.1 Identidades e `p_fundo`

In [10]:
erros = pd.DataFrame({j: c4.verificar_identidades(V4F[j]) for j in JOGOS})
display(erros.rename(columns=lambda j: str(j)).map("{:.2e}".format))
linhas_f = pd.concat([V4F[j]["linhas"].assign(jogo=j) for j in JOGOS], ignore_index=True)
display(Markdown("**Distribuição de `p_fundo`** (probabilidade de o passe chegar só com o fundo; os dois jogos juntos)"))
display(af.distribuicao_p_fundo(linhas_f))
for j in JOGOS:
    display(Markdown(f"**{j} ({NOMES[j]})**"))
    display(af.distribuicao_p_fundo(V4F[j]["linhas"]))

,1925299,1996435
I_mais_D_menos_v,3.12e-17,5.55e-17
shapley_I_menos_I,1.39e-17,2.78e-17
shapley_D_menos_D,6.94e-18,8.67e-18
soma_das_linhas_menos_v,2.78e-17,3.64e-17


**Distribuição de `p_fundo`** (probabilidade de o passe chegar só com o fundo; os dois jogos juntos)

,linhas,% com p_fundo < 0.9,% com p_fundo < 0.5,média,p10,mediana,p90
todas,121017.0000,45.6052,25.6790,0.7259,0.0726,0.9345,1.0000
recetor não marcado,92003.0000,34.3347,17.6799,0.8050,0.2027,0.9809,1.0000
recetor marcado,29014.0000,81.3435,51.0443,0.4750,0.0110,0.4790,0.9534


**1925299 (Brisbane Roar – Perth Glory)**

,linhas,% com p_fundo < 0.9,% com p_fundo < 0.5,média,p10,mediana,p90
todas,63151.0000,45.2139,25.3789,0.7277,0.0675,0.9383,1.0000
recetor não marcado,48312.0000,33.8922,17.3580,0.8069,0.1906,0.9831,1.0000
recetor marcado,14839.0000,82.0743,51.4927,0.4697,0.0102,0.4696,0.9510


**1996435 (Sydney FC – Adelaide United)**

,linhas,% com p_fundo < 0.9,% com p_fundo < 0.5,média,p10,mediana,p90
todas,57866.0000,46.0322,26.0066,0.7240,0.0784,0.9313,1.0000
recetor não marcado,43691.0000,34.8241,18.0358,0.8030,0.2137,0.9785,1.0000
recetor marcado,14175.0000,80.5785,50.5750,0.4806,0.0118,0.4896,0.9558


### 8.2 Totais por jogo, antes e depois

Somas sobre todos os momentos (xT esperado); `% com D < 0` e `% com v < 0` sobre os momentos com algum defesa em C.

In [11]:
MOMENTOS = {"sem fundo": {j: V4[j]["momentos"] for j in JOGOS}, "com fundo": {j: V4F[j]["momentos"] for j in JOGOS},
            "sem o guarda-redes": {j: V4G[j]["momentos"] for j in JOGOS}}
totais = af.antes_e_depois(MOMENTOS["sem fundo"], MOMENTOS["com fundo"], MOMENTOS["sem o guarda-redes"])
display(totais.sort_index(axis=1, level=0, sort_remaining=False)[[(j, v) for j in JOGOS for v in ("sem fundo", "com fundo", "com fundo, sem o guarda-redes")]])
display(Markdown("**Variante sem o guarda-redes em F, face à v4 com fundo completo** (diferença e variação, por jogo)"))
cmp_gr = pd.DataFrame({j: {"E(∅)": totais.loc["E(∅)", (j, "com fundo, sem o guarda-redes")] - totais.loc["E(∅)", (j, "com fundo")],
                           "I": totais.loc["I", (j, "com fundo, sem o guarda-redes")] - totais.loc["I", (j, "com fundo")],
                           "D": totais.loc["D", (j, "com fundo, sem o guarda-redes")] - totais.loc["D", (j, "com fundo")],
                           "v": totais.loc["v", (j, "com fundo, sem o guarda-redes")] - totais.loc["v", (j, "com fundo")],
                           "v / E(∅) (pontos %)": totais.loc["v / E(∅) (%)", (j, "com fundo, sem o guarda-redes")] - totais.loc["v / E(∅) (%)", (j, "com fundo")]}
                       for j in JOGOS})
display(cmp_gr)

jogo                        1925299                                          \
versão                    sem fundo com fundo com fundo, sem o guarda-redes   
momentos                  8915.0000 8915.0000                     8915.0000   
momentos com defesas em C 8276.0000 8276.0000                     8276.0000   
E(∅)                        46.5613   27.7410                       28.0078   
I                           16.1198    8.0062                        8.1744   
D                            4.2167    1.3714                        1.4268   
v                           20.3364    9.3776                        9.6011   
v / E(∅) (%)                43.6767   33.8039                       34.2803   
% com D < 0                 29.5191   45.7467                       45.7226   
% com v < 0                 18.7168   30.7516                       30.6670   

jogo                        1996435                                          
versão                    sem fundo com fundo com fundo, sem o guarda-redes  
momentos                  8145.0000 8145.0000                     8145.0000  
momentos com defesas em C 7600.0000 7600.0000                     7600.0000  
E(∅)                        40.7382   25.7960                       26.3715  
I                           10.7940    5.8319                        6.1760  
D                            1.7506    0.2881                        0.3154  
v                           12.5446    6.1200                        6.4914  
v / E(∅) (%)                30.7933   23.7247                       24.6152  
% com D < 0                 36.4342   47.7895                       47.8026  
% com v < 0                 22.2895   32.3421                       32.3026

**Variante sem o guarda-redes em F, face à v4 com fundo completo** (diferença e variação, por jogo)

,1925299,1996435
E(∅),0.2668,0.5755
I,0.1682,0.3441
D,0.0554,0.0273
v,0.2236,0.3714
v / E(∅) (pontos %),0.4763,0.8905


### 8.3 Lances: C06, C03, D01–D03 e W04, antes e depois

`D01`–`D03` e `W04` são os da 4.ª ronda anterior (sem fundo; índice em `outputs/cover_shadow/revisao_v4_sem_fundo/`). Shapley por defesa de I, D e total (decomposição simétrica); um defesa que só está em C numa das versões aparece com 0 na outra.

In [12]:
from footy.paths import output_dir

indice_antigo = pd.read_csv(output_dir("cover_shadow") / "revisao_v4_sem_fundo" / "indice_dos_lances.csv").set_index("lance")
LANCES = [("C06", 1925299, 12660), ("C03", 1996435, 39646)] + [(l, int(indice_antigo.loc[l, "jogo"]), int(indice_antigo.loc[l, "frame"]))
                                                           for l in ("D01", "D02", "D03", "W04")]
NOMES_JOGADORES = {j: JOGADORES[j].set_index("jogador")["nome"].to_dict() for j in JOGOS}
for rotulo, jogo, frame in LANCES:
    totais_lance, por_defesa = af.lance_antes_e_depois(V4[jogo], V4F[jogo], frame, NOMES_JOGADORES[jogo])
    display(Markdown(f"**{rotulo}** · jogo {jogo} · frame {frame}"))
    display(totais_lance)
    display(por_defesa)

**C06** · jogo 1925299 · frame 12660

,sem fundo,com fundo
E(∅),0.0029,0.0020
E(C),0.0026,0.0020
I,0.0002,0.0000
D,0.0001,-0.0000
v,0.0003,-0.0000


,I sem fundo,D sem fundo,total sem fundo,I com fundo,D com fundo,total com fundo
defesa,,,,,,
K. Jelacic,0.0002,0.0001,0.0003,0.0000,-0.0000,-0.0000


**C03** · jogo 1996435 · frame 39646

,sem fundo,com fundo
E(∅),0.0018,0.0013
E(C),0.0012,0.0007
I,0.0004,0.0004
D,0.0002,0.0002
v,0.0006,0.0006


,I sem fundo,D sem fundo,total sem fundo,I com fundo,D com fundo,total com fundo
defesa,,,,,,
P. Klimala,0.0004,0.0002,0.0006,0.0004,0.0002,0.0006


... (8 outputs left out)

,I sem fundo,D sem fundo,total sem fundo,I com fundo,D com fundo,total com fundo
defesa,,,,,,
Isaias,0.0000,-0.0002,-0.0002,0.0000,-0.0001,-0.0001
E. Alagich,0.0000,-0.0019,-0.0019,0.0000,-0.0010,-0.0010
D. Pierias,0.0000,-0.0028,-0.0028,0.0000,-0.0015,-0.0015
F. Talladira,0.0044,-0.0209,-0.0164,0.0043,-0.0093,-0.0050


**W04** · jogo 1996435 · frame 45252

,sem fundo,com fundo
E(∅),0.1467,0.0963
E(C),0.1676,0.1119
I,0.0022,0.0008
D,-0.0231,-0.0163
v,-0.0209,-0.0156


,I sem fundo,D sem fundo,total sem fundo,I com fundo,D com fundo,total com fundo
defesa,,,,,,
Léo Sena,0.0000,-0.0000,-0.0000,0.0000,-0.0000,-0.0000
C. Hollman,0.0004,-0.0047,-0.0043,0.0002,-0.0031,-0.0028
R. Grant,0.0006,-0.0066,-0.0060,0.0004,-0.0044,-0.0040
J. Courtney-Perkins,0.0013,-0.0118,-0.0105,0.0002,-0.0089,-0.0087


### 8.4 ρ de Spearman por jogador, dentro da posição

Ameaça negada por jogador (soma das médias por posse, xT) dos dois jogos juntos; ρ por posição (posições com pelo menos 4 jogadores), média ponderada e mínimo.

In [13]:
PRES = {j: DEFESAS[j][["frame", "posse_id", "jogador"]] for j in JOGOS}
sem_f = af.jogadores_de_uma_versao({j: V4[j]["momentos"] for j in JOGOS}, {j: V4[j]["atribuicoes"] for j in JOGOS}, PRES, JOGADORES)
com_f = af.jogadores_de_uma_versao({j: V4F[j]["momentos"] for j in JOGOS}, {j: V4F[j]["atribuicoes"] for j in JOGOS}, PRES, JOGADORES)
v3_j = af.jogadores_v3({j: av.ler_variante(j, "base", av.PASTA_AVALIACAO_V3) for j in JOGOS}, PRES, JOGADORES)
for rotulo, a, b in (("v4 sem fundo vs v4 com fundo", sem_f, com_f), ("v3 vs v4 com fundo", v3_j, com_f), ("v3 vs v4 sem fundo (referência)", v3_j, sem_f)):
    r = af.rho_entre_versoes(a, b)
    display(Markdown(f"**ρ de Spearman, {rotulo}** · média ponderada {r['rho_media']:.3f} · mínimo {r['rho_min']:.3f}"))
    display(r["posicoes"].set_index("posicao"))

**ρ de Spearman, v4 sem fundo vs v4 com fundo** · média ponderada 0.910 · mínimo 0.762

,n,rho
posicao,,
AM,5,1.0000
CF,9,0.9333
LB,4,0.8000
LCB,5,0.9000
LW,8,0.7619
RB,5,1.0000
RCB,4,0.8000
RDM,5,1.0000
RW,8,0.9762


**ρ de Spearman, v3 vs v4 com fundo** · média ponderada 0.736 · mínimo 0.000

,n,rho
posicao,,
AM,5,1.0000
CF,9,0.9333
LB,4,0.8000
LCB,5,0.9000
LW,8,0.3571
RB,5,0.9000
RCB,4,0.0000
RDM,5,0.7000
RW,8,0.8810


**ρ de Spearman, v3 vs v4 sem fundo (referência)** · média ponderada 0.827 · mínimo 0.400

,n,rho
posicao,,
AM,5,1.0000
CF,9,0.9333
LB,4,1.0000
LCB,5,1.0000
LW,8,0.5476
RB,5,0.9000
RCB,4,0.4000
RDM,5,0.7000
RW,8,0.9286
